## 6 — MRdeeP (state-level estimates, CES sample2 / 3000)
Multivariate Multilevel Regression with Deep Generative Post-Stratification.
All 7 climate opinion outcomes estimated simultaneously:
`climate_problem`, `regulate_carbon`, `renewable_fuel`, `clean_air_water`,
`fuel_efficiency`, `fossil_fuel`, `paris_agreement`.

Pipeline:
1. `insert_data` — encodes CES survey + county-level benchmark
2. `fit` — trains an ensemble of CGANs (Wasserstein loss + gradient penalty)
3. `post_stratify('state_fips')` — generates synthetic micro-data per demographic
   cell, groups by state → extracts all outcome estimates

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.environ['DEEPVERSE_BACKEND'] = 'pytorch'
sys.path.insert(0, '/Users/carmenk/Documents/CSS/Capstone/mrdeep-main/python')
from mrdeep import MRdeeP

sys.path.insert(0, str(Path('.').resolve()))
from utils import OUTPUT_DIR, STATE_FIPS_TO_NAME, SURVEY_PATH, save_estimates

DATA_DIR   = Path("../../")
OUTCOME    = ['climate_problem', 'regulate_carbon', 'renewable_fuel',
              'clean_air_water', 'fuel_efficiency', 'fossil_fuel',
              'paris_agreement']
MODEL_NAME = 'mrdeep'

### 1. Load and prepare data

In [2]:
OUTCOME_COLS = ['climate_problem','regulate_carbon','renewable_fuel',
                'clean_air_water','fuel_efficiency','fossil_fuel','paris_agreement']
DEMOG_VARS   = ['gender', 'race4', 'educ_category', 'county_fips', 'state_fips']

raw       = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str, 'county_fips': str})
ps_county = pd.read_csv(DATA_DIR / 'post_stratification_frame' / 'poststrat_county.csv',
                        dtype={'state_fips': str, 'county_fips': str})

raw['county_fips'] = raw['county_fips'].astype(str).str.zfill(5)
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in raw.columns]

survey = raw[DEMOG_VARS + OUTCOME_COLS].dropna().copy()
survey['educ_category'] = survey['educ_category'].astype(str)

benchmark = ps_county[DEMOG_VARS + ['N_rounded']].copy()
benchmark['educ_category'] = benchmark['educ_category'].astype(str)
target_rows = len(benchmark)
benchmark['count'] = np.maximum(
    1,
    (benchmark['N_rounded'] / benchmark['N_rounded'].sum() * target_rows).round(),
).astype(int)
benchmark = benchmark.drop(columns=['N_rounded'])

print(f'Survey (complete cases): {len(survey):,}')
for oc in OUTCOME_COLS:
    print(f'  {oc}: {survey[oc].mean()*100:.1f}% support')
print(f'Benchmark strata: {len(benchmark):,}  augmented rows: {benchmark["count"].sum():,}')

Survey (complete cases): 2,977
  climate_problem: 62.8% support
  regulate_carbon: 65.1% support
  renewable_fuel: 60.0% support
  clean_air_water: 56.5% support
  fuel_efficiency: 65.9% support
  fossil_fuel: 62.9% support
  paris_agreement: 58.6% support
Benchmark strata: 99,940  augmented rows: 170,690


### 2. Insert data into MRdeeP

In [ ]:
mod = MRdeeP(ensembles=5, random_state=42)

mod.insert_data(
    survey     = survey,
    benchmark  = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = 'count',
    oversample = 1,
)
print(mod)

### 3. Train CGAN ensemble
Wasserstein GAN with gradient penalty. Factorized embeddings give per-variable
demographic embeddings (race, educ, gender, county, state) instead of a single
stratum embedding — critical for cross-stratum pooling with sparse cells.

In [ ]:
mod.fit(
    # architecture
    k                          = 32,
    gan_type                   = 'cganwl',
    embed_dim                  = 50,
    neurons_generator          = (256, 256, 256, 256),
    neurons_critic             = (192, 192, 192, 192),
    activation_generator       = 'relu',
    activation_critic          = 'relu',
    final_activation_generator = 'sigmoid',
    dropout_generator          = 0.0,
    dropout_critic             = 0.2,
    bn_momentum                = 0.1,
    # WGAN-GP
    critic_steps               = 5,
    gp_weight                  = 10.0,
    learning_rate              = (5e-5, 1e-4),
    # training
    epochs                     = 1000,
    batch_size                 = 500,
    patience                   = 100,
    validation_split           = 0.15,
    # factorized per-variable embeddings
    factorized_embed           = True,
    # ensemble post-processing
    calibrate                  = True,
    reject_collapsed           = True,
    reject_threshold           = 0.15,
    aggregate                  = 'median',
    # disable auto-tune (use explicit params above)
    auto_tune                  = False,
    print_runtime              = True,
)
print(mod)

### 4. Post-stratify → state-level estimates for all 7 outcomes

In [ ]:
estimates = mod.post_stratify(levels='state_fips', se=True)
print(f'Estimates shape: {estimates.shape}  ({estimates["state_fips"].nunique()} states)')
estimates.head()

### 5. Save all outcome estimates

In [ ]:
for OUTCOME_VAR in OUTCOME:
    result = estimates[['state_fips', OUTCOME_VAR]].rename(
        columns={OUTCOME_VAR: 'estimate'}
    ).copy()
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)

    save_estimates(result, MODEL_NAME, OUTCOME_VAR)

    print(f'\n--- {OUTCOME_VAR} ---')
    print(f'National mean: {result["estimate"].mean():.3f}')
    print(f'State range:   {result["estimate"].min():.3f} – {result["estimate"].max():.3f}')
    print(result.sort_values("estimate", ascending=False).head(5)[["state_name","estimate"]].to_string(index=False))